# FAISS 임베딩 인덱스 구축

Hugging Face 데이터셋을 다운로드하여 마크다운 섹션별로 청킹하고, multilingual-e5-large-instruct 모델로 임베딩한 후 FAISS 인덱스로 저장합니다.


## 1. 환경 설정 및 라이브러리 임포트


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from datasets import load_dataset
from tqdm import tqdm
import logging
import importlib.util
import os

# 프로젝트 루트 경로 찾기 (절대 경로 사용)
project_root = Path("/data/ephemeral/home/T8001/pro-nlp-generationfornlp-nlp-07")

# 프로젝트 루트로 작업 디렉토리 변경
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"Current working directory: {os.getcwd()}")
print(f"Project root: {project_root}")
print(f"Project root exists: {project_root.exists()}")

# 설정 임포트
from src.config.config import (
    FAISS_INDEX_DIR,
    EMBEDDING_MODEL_NAME,
    EMBEDDING_MAX_TOKENS,
    CHUNKING_HEADERS,
    CHUNK_SIZE,
    CHUNK_OVERLAP,
    EMBEDDING_BATCH_SIZE,
    EMBEDDING_DEVICE
)

# ke-02 모듈 임포트 (디렉토리 이름에 하이픈이 있어서 importlib 사용)
ke02_path = project_root / "scripts" / "experiments" / "ke-02"
sys.path.insert(0, str(ke02_path))

# 모듈 임포트
from markdown_chunker import (
    chunk_markdown_document,
    prepare_embedding_text,
    chunk_dataset
)
from embedder import create_embedder
from faiss_indexer import FAISSIndexer

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"Project root: {project_root}")
print(f"FAISS index directory: {FAISS_INDEX_DIR}")


Current working directory: /data/ephemeral/home/T8001/pro-nlp-generationfornlp-nlp-07
Project root: /data/ephemeral/home/T8001/pro-nlp-generationfornlp-nlp-07
Project root exists: True


Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu128 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


Project root: /data/ephemeral/home/T8001/pro-nlp-generationfornlp-nlp-07
FAISS index directory: /data/ephemeral/home/T8001/pro-nlp-generationfornlp-nlp-07/data/faiss_index


## 2. Hugging Face 데이터셋 다운로드


In [2]:
dataset_name = "NLP-07-ODQA/kowiki-cleaned"

print(f"Loading dataset: {dataset_name}")
dataset = load_dataset(dataset_name)

Loading dataset: NLP-07-ODQA/kowiki-cleaned


In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['content', 'title', 'page_id', 'text_length'],
        num_rows: 565484
    })
})

In [5]:
# train split 사용 (또는 적절한 split 선택)
data = dataset['train']

In [6]:
# 샘플 데이터 확인
sample = data[0]
for key, value in sample.items():
    print(f"{key}: {value}")

content: 서훈 = 노벨 평화상(2002) **제임스 얼 "지미 카터 주니어"**(James Earl Jimmy Carter, Jr., 1924년 10월 1일~2024년 12월 29일)는 미국의 제39대 대통령 (1977-81)을 지낸 미국의 정치인이다.https://www.washingtonpost.com/obituaries/2024/12/29/jimmy-carter-president-dead/확인날짜=29 December 2024 민주당 소속으로 1963년부터 1967년까지 조지아주 상원 의원, 1971년부터 1975년까지 조지아주의 76대 주지사을 지냈다. 카터는 100세까지 산 최초의 대통령으로 미국 역사상 가장 장수한 대통령이다. 카터는 조지아주 플레인스에서 태어나고 자랐다. 1946년 미국 해군사관학교를 졸업하고 미국 해군 잠수함에 승선했다. 카터는 군 복무를 마치고 고향으로 돌아와 가족의 땅콩 재배 사업을 되살렸다. 카터는 인종 분리 정책에 반대하며 성장하던 민권 운동을 지지했고, 민주당 내에서 활동가가 되었다. 1963년부터 1967년까지 조지아주 상원 의원으로 재직하였고, 1971년부터 1975년까지 조지아 주지사로 재직했다. 조지아 주 밖에서는 잘 알려지지 않은 다크호스 후보였던 카터는 민주당 후보로 지명되어 1976년 대선에서 공화당의 현직 대통령인 제럴드 포드를 상대로 신승했다. 카터는 취임 둘째 날 베트남 전쟁에서 병역을 기피한 모든 사람들을 사면했다. 에너지부와 교육부를 설립했으며, 에너지 절약, 가격 통제, 신기술을 포함한 국가 에너지 정책을 만들었다. 스태그플레이션에 대응하는 동시에 카터는 캠프 데이비드 협정, 파나마 운하 조약, 제2차 전략 무기 제한 협상을 성공적으로 추진했다. 하지만 임기 말에는 이란 인질 사태, 에너지 위기, 스리마일 섬 사고, 니카라과 혁명, 그리고 소련의 아프가니스탄 침공 등 위기가 이어졌다. 아프가니스탄 침공에 대응하여 카터는 데탕트 정책을 종식시키고 카터 독트린을 선포했으며, 소련에 곡물 금수조치

## 3. 청킹 예시 확인 (샘플 문서)


In [8]:
# 샘플 문서 선택 (1-2개)
sample_indices = [0, 1]
sample_docs = [data[idx] for idx in sample_indices]

all_sample_chunks = []

for sample_doc in sample_docs:
    context = sample_doc.get('content', '')  # 데이터셋 컬럼명이 'content'
    title = sample_doc.get('title', '')
    page_id = sample_doc.get('page_id', '')
    
    print(f"\n{'='*80}")
    print(f"Title: {title}")
    print(f"Page ID: {page_id}")
    print(f"Context length: {len(context)} characters")
    print(f"{'='*80}")
    
    # 청킹 실행
    chunks = chunk_markdown_document(
        context=context,
        title=title,
        page_id=page_id,
        headers_to_split_on=CHUNKING_HEADERS,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )
    
    all_sample_chunks.extend(chunks)
    
    print(f"\nNumber of chunks: {len(chunks)}")
    
    # 청킹 결과 시각화
    chunk_data = []
    for i, chunk in enumerate(chunks):
        header_path = []
        for level in ["Header 1", "Header 2", "Header 3"]:
            if level in chunk.metadata:
                header_path.append(chunk.metadata[level])
        
        header_str = " > ".join(header_path) if header_path else "(No header)"
        estimated_tokens = int(len(chunk.page_content) * 1.2)  # 대략적인 토큰 수
        
        chunk_data.append({
            'chunk_id': i,
            'title': chunk.metadata.get('title', ''),
            'page_id': chunk.metadata.get('page_id', ''),
            'header_path': header_str,
            'text_length': len(chunk.page_content),
            'estimated_tokens': estimated_tokens,
            'within_limit': estimated_tokens <= EMBEDDING_MAX_TOKENS,
            'preview': chunk.page_content[:100] + '...' if len(chunk.page_content) > 100 else chunk.page_content
        })
    
    # DataFrame으로 표시
    df_chunks = pd.DataFrame(chunk_data)
    print("\nChunking Results:")
    print(df_chunks.to_string(index=False))
    
    # 각 청크의 내용 미리보기
    print("\n" + "="*80)
    print("Chunk Details:")
    print("="*80)
    for i, chunk in enumerate(chunks[:3]):  # 처음 3개만 표시
        print(f"\n--- Chunk {i+1} ---")
        print(f"Metadata: {chunk.metadata}")
        print(f"Content (first 200 chars): {chunk.page_content[:200]}...")
        
        # 임베딩용 텍스트 생성
        embedding_text = prepare_embedding_text(chunk)
        print(f"\nEmbedding text (first 200 chars): {embedding_text[:200]}...")
        print(f"Embedding text length: {len(embedding_text)} characters")

print(f"\n\nTotal sample chunks: {len(all_sample_chunks)}")



Title: 지미 카터
Page ID: 5
Context length: 14098 characters

Number of chunks: 47

Chunking Results:
 chunk_id title  page_id                                   header_path  text_length  estimated_tokens  within_limit                                                                                                 preview
        0 지미 카터        5                                  Introduction          398               477          True 서훈 = 노벨 평화상(2002) **제임스 얼 "지미 카터 주니어"**(James Earl Jimmy Carter, Jr., 1924년 10월 1일~2024년 12월 29일)는 미...
        1 지미 카터        5                                  Introduction          398               477          True 대통령이다. 카터는 조지아주 플레인스에서 태어나고 자랐다. 1946년 미국 해군사관학교를 졸업하고 미국 해군 잠수함에 승선했다. 카터는 군 복무를 마치고 고향으로 돌아와 가족의 땅...
        2 지미 카터        5                                  Introduction          397               476          True 기피한 모든 사람들을 사면했다. 에너지부와 교육부를 설립했으며, 에너지 절약, 가격 통제, 신기술을 포함한 국가 에너지 정책을 만들었다. 스태그플레이션에 대응하는 동시에 카터는 캠...
        3 지미 카터      

## 4. 전체 데이터셋 청킹

In [10]:
print("Starting chunking for entire dataset...")
print(f"Total documents: {len(data)}")

all_chunks = []

for idx, row in enumerate(tqdm(data, desc="Chunking documents")):
    context = row.get('content', '')  # 데이터셋 컬럼명이 'content'
    title = row.get('title', '')
    page_id = row.get('page_id', '')
    
    if not context or not context.strip():
        continue
    
    chunks = chunk_markdown_document(
        context=context,
        title=title,
        page_id=page_id,
        headers_to_split_on=CHUNKING_HEADERS,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )
    
    all_chunks.extend(chunks)
    
    # if (idx + 1) % 1000 == 0:
    #     print(f"Processed {idx + 1} documents, total chunks: {len(all_chunks)}")

print(f"\nChunking completed!")
print(f"Total documents: {len(data)}")
print(f"Total chunks: {len(all_chunks)}")
print(f"Average chunks per document: {len(all_chunks) / len(data):.2f}")


Starting chunking for entire dataset...
Total documents: 565484


Chunking documents: 100%|██████████| 565484/565484 [08:34<00:00, 1098.66it/s]


Chunking completed!
Total documents: 565484
Total chunks: 4322674
Average chunks per document: 7.64


### 4-1. 청킹 결과를 Hugging Face 데이터셋으로 업로드

In [11]:
# 청킹된 데이터를 Hugging Face Dataset 형식으로 변환
from datasets import Dataset
from src.config.config import HF_ORG, HF_TOKEN

print("Converting chunks to Hugging Face Dataset format...")

# 청크 데이터를 딕셔너리 리스트로 변환
chunk_data = []
for chunk in all_chunks:
    # 헤더 경로 생성
    header_path = []
    for level in ["Header 1", "Header 2", "Header 3"]:
        if level in chunk.metadata:
            header_path.append(chunk.metadata[level])
    
    chunk_dict = {
        "page_content": chunk.page_content,
        "title": chunk.metadata.get("title", ""),
        "page_id": chunk.metadata.get("page_id", ""),
        "header_1": chunk.metadata.get("Header 1", ""),
        "header_2": chunk.metadata.get("Header 2", ""),
        "header_3": chunk.metadata.get("Header 3", ""),
        "header_path": " > ".join(header_path) if header_path else "",
        "text_length": len(chunk.page_content),
    }
    chunk_data.append(chunk_dict)

print(f"Total chunks to upload: {len(chunk_data)}")

# Dataset 생성
chunk_dataset = Dataset.from_list(chunk_data)

print(f"Dataset created with {len(chunk_dataset)} samples")
print(f"Dataset features: {chunk_dataset.features}")

Converting chunks to Hugging Face Dataset format...
Total chunks to upload: 4322674
Dataset created with 4322674 samples
Dataset features: {'page_content': Value('string'), 'title': Value('string'), 'page_id': Value('int64'), 'header_1': Value('string'), 'header_2': Value('string'), 'header_3': Value('string'), 'header_path': Value('string'), 'text_length': Value('int64')}


In [12]:
# Hugging Face에 데이터셋 업로드
from huggingface_hub import login
import os

# Hugging Face 로그인
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face")
else:
    print("Warning: HF_TOKEN not found. Please set it in .env file or environment variable.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face


In [13]:
# 데이터셋 이름 설정
dataset_name = f"{HF_ORG}/kowiki-cleaned-chunked-markdown-400"
print(f"\nUploading dataset to: {dataset_name}")

# 데이터셋 업로드
try:
    chunk_dataset.push_to_hub(
        repo_id=dataset_name,
        private=False,  # public으로 설정 (필요시 True로 변경)
        commit_message="Add chunked kowiki dataset with markdown section-based chunking / branch : ke-02"
    )
    print(f"\n✅ Dataset uploaded successfully!")
    print(f"Dataset URL: https://huggingface.co/datasets/{dataset_name}")
except Exception as e:
    print(f"\n❌ Error uploading dataset: {e}")
    print("Please check your HF_TOKEN and try again.")



Uploading dataset to: NLP-07-ODQA/kowiki-cleaned-chunked-markdown-400


Uploading the dataset shards:   0%|          | 0/6 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✅ Dataset uploaded successfully!
Dataset URL: https://huggingface.co/datasets/NLP-07-ODQA/kowiki-cleaned-chunked-markdown-400


## 5-0. Hugging Face에서 청킹된 데이터셋 로드 (선택사항)

커널이 꺼져서 이전 청킹 결과를 잃어버린 경우, 이미 업로드한 데이터셋을 로드하여 사용할 수 있습니다.

In [2]:
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

In [3]:
# Hugging Face에서 청킹된 데이터셋 로드
from langchain_core.documents import Document
from src.config.config import HF_ORG, HF_TOKEN 

# 이미 업로드한 데이터셋 이름
chunked_dataset_name = f"{HF_ORG}/kowiki-cleaned-chunked-markdown-400"

print(f"Loading chunked dataset from Hugging Face: {chunked_dataset_name}")

# 데이터셋 로드
try:
    chunked_dataset = load_dataset(chunked_dataset_name, split="train")
    print(f"✅ Dataset loaded successfully!")
    print(f"Total chunks: {len(chunked_dataset)}")
    print(f"Dataset features: {chunked_dataset.features}")
    
    # Dataset을 Document 객체 리스트로 변환
    print("\nConverting dataset to Document objects...")
    all_chunks = []
    
    for item in tqdm(chunked_dataset, desc="Converting to Documents"):
        # 메타데이터 구성
        metadata = {
            "title": item.get("title", ""),
            "page_id": item.get("page_id", ""),
        }
        
        # 헤더 정보 추가 (비어있지 않은 경우만)
        if item.get("header_1"):
            metadata["Header 1"] = item["header_1"]
        if item.get("header_2"):
            metadata["Header 2"] = item["header_2"]
        if item.get("header_3"):
            metadata["Header 3"] = item["header_3"]
        
        # Document 객체 생성
        doc = Document(
            page_content=item.get("page_content", ""),
            metadata=metadata
        )
        all_chunks.append(doc)
    
    print(f"\n✅ Conversion completed!")
    print(f"Total Document objects: {len(all_chunks)}")
    print(f"Sample chunk metadata: {all_chunks[0].metadata if all_chunks else 'N/A'}")
    
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    print("Please check the dataset name and try again.")
    print("If the dataset doesn't exist, you need to run the chunking section first.")


Loading chunked dataset from Hugging Face: NLP-07-ODQA/kowiki-cleaned-chunked-markdown-400
✅ Dataset loaded successfully!
Total chunks: 4322674
Dataset features: {'page_content': Value('string'), 'title': Value('string'), 'page_id': Value('int64'), 'header_1': Value('string'), 'header_2': Value('string'), 'header_3': Value('string'), 'header_path': Value('string'), 'text_length': Value('int64')}

Converting dataset to Document objects...


Converting to Documents: 100%|██████████| 4322674/4322674 [06:03<00:00, 11879.32it/s]


✅ Conversion completed!
Total Document objects: 4322674
Sample chunk metadata: {'title': '지미 카터', 'page_id': 5, 'Header 1': 'Introduction'}


**참고**: 위 셀을 실행하면 `all_chunks` 변수가 생성됩니다. 
이제 "5. 임베딩 생성" 섹션부터 진행하면 됩니다.


## 5. 임베딩 생성 (GPU 배치 처리)

In [4]:
# Embedder 생성
print(f"Creating embedder with model: {EMBEDDING_MODEL_NAME}")
print(f"Device: {EMBEDDING_DEVICE}, Batch size: {EMBEDDING_BATCH_SIZE}")

embedder = create_embedder(
    model_name=EMBEDDING_MODEL_NAME,
    device=EMBEDDING_DEVICE,
    batch_size=EMBEDDING_BATCH_SIZE,
    max_length=EMBEDDING_MAX_TOKENS
)

print(f"Embedding dimension: {embedder.get_embedding_dim()}")

INFO:embedder:Loading embedding model: intfloat/multilingual-e5-large-instruct
INFO:embedder:Device: cuda, Batch size: 32, Max length: 512
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: intfloat/multilingual-e5-large-instruct


Creating embedder with model: intfloat/multilingual-e5-large-instruct
Device: cuda, Batch size: 32


/data/ephemeral/home/py310/lib/python3.10/site-packages/huggingface_hub/file_download.py:798: UserWarning: Not enough free disk space to download the file. The expected file size is: 0.00 MB. The target location /data/ephemeral/home/.cache/huggingface/hub/models--intfloat--multilingual-e5-large-instruct/blobs only has 0.00 MB free disk space.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

...

/data/ephemeral/home/py310/lib/python3.10/site-packages/huggingface_hub/file_download.py:798: UserWarning: Not enough free disk space to download the file. The expected file size is: 17.08 MB. The target location /data/ephemeral/home/.cache/huggingface/hub/models--intfloat--multilingual-e5-large-instruct/blobs only has 0.00 MB free disk space.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

Embedding dimension: 1024


In [ ]:
# 임베딩용 텍스트 준비
print("Preparing embedding texts...")
embedding_texts = [prepare_embedding_text(chunk) for chunk in tqdm(all_chunks, desc="Preparing texts")]

print(f"Total texts to embed: {len(embedding_texts)}")
print(f"Sample text length: {len(embedding_texts[0]) if embedding_texts else 0} characters")


In [ ]:
# 임베딩 생성
print("Generating embeddings...")
embeddings = embedder.encode_texts(
    texts=embedding_texts,
    show_progress_bar=True
)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")


## 6. FAISS 인덱스 구축 및 저장

In [ ]:
# FAISS 인덱서 생성
print("Creating FAISS index...")
indexer = FAISSIndexer(
    embedding_dim=embeddings.shape[1],
    index_type="cosine"  # 코사인 유사도 사용
)

# 문서와 임베딩 추가
print("Adding embeddings to index...")
indexer.add_documents(embeddings, all_chunks)

# 통계 정보
stats = indexer.get_stats()
print(f"\nIndex statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")


In [ ]:
# 인덱스 저장
FAISS_INDEX_DIR.mkdir(parents=True, exist_ok=True)

index_path = FAISS_INDEX_DIR / "faiss_index.bin"
metadata_path = FAISS_INDEX_DIR / "metadata.json"

print(f"Saving index to: {index_path}")
indexer.save(index_path, metadata_path)

print(f"\nIndex saved successfully!")
print(f"Index file: {index_path}")
print(f"Metadata file: {metadata_path}")


## 7. 인덱스 검증 (테스트 검색)

In [ ]:
# 테스트 쿼리로 검색
test_query = "한국의 역사"

print(f"Test query: {test_query}")

# 쿼리 임베딩 생성
query_text = f"query: {test_query}"  # e5 모델의 query 포맷
query_embedding = embedder.encode_texts([query_text], show_progress_bar=False)

# 검색
results = indexer.search(query_embedding[0], k=5)

print(f"\nTop 5 search results:")
print("="*80)
for i, result in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(f"  Distance: {result['distance']:.4f}")
    print(f"  Title: {result['metadata'].get('title', 'N/A')}")
    print(f"  Page ID: {result['metadata'].get('page_id', 'N/A')}")
    
    # Header 경로
    header_path = []
    for level in ["Header 1", "Header 2", "Header 3"]:
        if level in result['metadata']:
            header_path.append(result['metadata'][level])
    if header_path:
        print(f"  Header path: {' > '.join(header_path)}")
    
    content = result['metadata'].get('page_content', '')
    print(f"  Content preview: {content[:200]}...")
